In [ ]:
import pandas as pd
import plotly.express as px
from scipy.stats import spearmanr

from utils import get_recession_data, get_recession_start_end_list

In [ ]:
fed_funds = pd.read_csv("data/fed_funds.csv").dropna()
fed_funds["date"] = pd.to_datetime(fed_funds["date"])
fed_funds.set_index("date", inplace=True)
fed_funds

In [ ]:
gold = pd.read_csv("data/WPU10210501.csv").dropna()
gold["date"] = pd.to_datetime(gold["date"])
gold = gold.set_index("date")
gold

In [ ]:
# Ensure df1 and df2 have the same dates in their indices
common_dates = gold.index.intersection(fed_funds.index)
fed_funds = fed_funds.reindex(common_dates)
gold = gold.reindex(common_dates)

if not fed_funds.index.equals(gold.index):
    raise ValueError("DataFrames do not have the same dates in their indices")

In [ ]:
df = pd.merge(
    fed_funds["interest rate"], gold["gold price"], left_index=True, right_index=True
)

In [ ]:
df.corr("spearman")

In [ ]:
corr_df = df["interest rate"].rolling(window=60).corr(df["gold price"])
corr_df

In [ ]:
# normalize each row of data in df except for the index and correlation
df = (df - df.min()) / (df.max() - df.min())

df["correlation"] = corr_df
df = df.dropna()
df.reset_index(inplace=True)

recession_df = get_recession_data()
recession_df = recession_df.reindex(common_dates)

In [ ]:
fig = px.line(
    df,
    x="date",
    y=df.columns,
    hover_data={"date": "|%B %d, %Y"},
    title="Correlation between Gold Prices and Interest Rates",
    template="plotly_dark",
    width=1200,
    height=600,
)

for row in get_recession_start_end_list(recession_df):
    x0 = str(row[0].date())
    x1 = str(row[1].date())

    fig.add_vrect(
        x0=x0,
        x1=x1,
        fillcolor="red",
        opacity=0.25,
        line_width=0,
    )

fig.show()

In [ ]:
"""Double check the correlation using a different method"""

spear_corr, _ = spearmanr(df["gold price"], df["interest rate"])
print("Spearmans correlation: %.2f" % spear_corr)